In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [9]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import requests

In [10]:
@tool
def get_conversion_rate(base_currency:str,target_currency:str)->float:
    """ 
        this function fetches the conversion rate between the base currency and the target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/8b04616be8f3a801502b3436/pair/{base_currency}/{target_currency}'
    result = requests.get(url=url)
    return result


In [11]:
@tool
def convert(base_currency:str,conversion_rate:float)->float:
    """ 
        this tool convert the basecurrency into the target currency
    """
    return base_currency * conversion_rate

In [12]:
llm = ChatOpenAI()

In [14]:
llm_with_tools = llm.bind_tools([convert,get_conversion_rate])

In [23]:
messages=[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 10 USD to NPR')]

In [24]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 10 USD to NPR', additional_kwargs={}, response_metadata={})]

In [25]:
llm_with_tools.invoke(messages).tool_calls

[{'name': 'get_conversion_rate',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'call_QQUhVLZ79JPV5rFA0wdhqXIh',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency': 'USD', 'conversion_rate': 118.09},
  'id': 'call_7OvzrANONhJ7uaRKNLIJ7xTI',
  'type': 'tool_call'}]

### you may see in the above output the conversion_rate is prefetched which is wrong llm based on previous parametric data fetched that so to solve this their is a concept of  injectedToolArgs in langchain instead of tools

In [94]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [95]:
@tool
def get_conversion_rate1(base_currency:str,target_currency:str)->float:
    """ 
        this function fetches the conversion rate between the base currency and the target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/8b04616be8f3a801502b3436/pair/{base_currency}/{target_currency}'
    result = requests.get(url=url)
    return result.json()


In [96]:
@tool
def convert1(base_currency:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """ 
        this tool convert the basecurrency into the target currency
    """
    return base_currency * conversion_rate

In [97]:
# you may see below no argument named conversion_rate is shown there
convert1.args

{'base_currency': {'title': 'Base Currency', 'type': 'integer'}}

In [98]:
model = ChatOpenAI()

In [99]:
model_with_tools = model.bind_tools([convert1,get_conversion_rate1])

In [100]:
chats =[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 10 USD to NPR')]

In [101]:
chats

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 10 USD to NPR', additional_kwargs={}, response_metadata={})]

In [102]:
ai_message = model_with_tools.invoke(chats)

In [103]:
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 111, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000135, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000135, 'upstream_inference_prompt_cost': 5.55e-05, 'upstream_inference_completions_cost': 7.95e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1783334037-0AbDT2a56uOnuEzmKCPY', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f36fd-e301-7ea0-8828-cef5d3eb510f-0', tool_calls=[{'name': 'get_conversion_rate1', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'call_0ZzBmP4gYZI4tzvUvK9o

In [104]:
ai_message.tool_calls

[{'name': 'get_conversion_rate1',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'call_0ZzBmP4gYZI4tzvUvK9ovt0k',
  'type': 'tool_call'},
 {'name': 'convert1',
  'args': {'base_currency': 10},
  'id': 'call_NIeQ32k4CqCBKzZkyjAe0wYh',
  'type': 'tool_call'}]

In [105]:
chats.append(ai_message)

In [106]:
chats

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 10 USD to NPR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 111, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000135, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000135, 'upstream_inference_prompt_cost': 5.55e-05, 'upstream_inference_completions_cost': 7.95e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1783334037-0AbDT2a56uOnuEzmKCPY', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f36fd

In [109]:
import json
for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name']=='get_conversion_rate1':
        tool_message1 = get_conversion_rate1.invoke(tool_call)
        # fetch this conversion rate
        # type(tool_message1.content) ->str so convert into dict  using json.loads() deserialization
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        chats.append(tool_message1)
    # place the conversion rate into convert function args
    if tool_call['name']=='convert1':
        tool_call['args']['conversion_rate']=conversion_rate
        tool_message2 = convert1.invoke(tool_call)
        chats.append(tool_message2)

In [110]:
chats

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that can you convert 10 USD to NPR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 111, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.000135, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.000135, 'upstream_inference_prompt_cost': 5.55e-05, 'upstream_inference_completions_cost': 7.95e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-3.5-turbo', 'system_fingerprint': None, 'id': 'gen-1783334037-0AbDT2a56uOnuEzmKCPY', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f36fd

In [112]:
llm_with_tools.invoke(chats).content

'The conversion rate between USD and NPR is 152.5406. Based on this conversion rate, 10 USD is equivalent to 1525.41 NPR.'